In [1]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 加载环境变量（名为.env)的文件，参数是文件路径，当不加参数时，会从脚本所在目录开始向上级查找
load_dotenv()

model = ChatDeepSeek(
    model="deepseek-flash",
    api_base="https://api.deepseek.com",
    temperature=0, # 非思考模式需要降低随机性
    max_tokens=2048, # 这个 tokens 是输入的还是输出的？ # 指的是最多生成多少输出 token，不设输入长度上限
    timeout=30,
    max_retries=0, # 失败时直接观察失败原因
    extra_body={"thinking": {"type": "disabled"}}
)

INIT_MESSAGE = "请只回复：连接成功"

response = model.invoke(INIT_MESSAGE)
print("模型回复：", response.content)
print("Token 用量：", response.usage_metadata)

模型回复： 连接成功
Token 用量： {'input_tokens': 10, 'output_tokens': 2, 'total_tokens': 12, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [2]:
import pandas as pd


dev_results =  pd.read_json("./results/b0_dev_results.jsonl", lines=True)
sources = pd.read_json("./data/sources.json")
evidence_df = pd.read_json("./data/evidence.jsonl", lines=True)

# print(dev_results)
# print(sources)
# print(evidence_df)

"""
evidence_ids: 本次选择的有序 ID 列表，先取检索结果前三个
evidence_df: 证据表
sources: 来源记录列表
"""

def build_context(evidence_ids, evidence_df, sources):
    cards = ""
    for evidence_id in evidence_ids:
        evidence = evidence_df[evidence_df["id"] == evidence_id].iloc[0]
        source = sources[sources["id"] == evidence["source_id"]].iloc[0]
        card = f"{evidence_id}\n发布机构: {source["publisher"]}\n文件名: {source["title"]}\n定位: {evidence["locator"]}\n原文链接: {source["url"]}\n证据正文（项目转述）: {evidence["text"]}"
        cards += f"{card}\n\n"
    return cards
    
rag_case = dev_results[dev_results["case_id"] == "DEV-01"].iloc[0]
selected_ids = rag_case["retrieved_ids"][:3]
evidence_context = build_context(selected_ids, evidence_df, sources)

print("本次证据数量：", len(selected_ids))
print("本次证据 ID：", selected_ids)
print(evidence_context)

本次证据数量： 3
本次证据 ID： ['F_SCOPE', 'AC_INCIDENTAL', 'AC_RADIOGRAPH']
F_SCOPE
发布机构: Fleischner Society / Radiology
文件名: Guidelines for Management of Incidental Pulmonary Nodules Detected on CT Images (2017)
定位: Introduction and Implications for Patient Care
原文链接: https://pubs.rsna.org/doi/full/10.1148/radiol.2017161659
证据正文（项目转述）: Fleischner 2017 addresses pulmonary nodules found incidentally on CT in adults aged 35 years or older. Lung cancer screening CT is a separate setting with its own reporting framework.

AC_INCIDENTAL
发布机构: American College of Radiology
文件名: ACR Appropriateness Criteria: Incidentally Detected Indeterminate Pulmonary Nodule
定位: Variants 2 and 3, incidental nodules on chest CT
原文链接: https://acsearch.acr.org/docs/69455/Narrative/
证据正文（项目转述）: ACR Appropriateness Criteria for incidental indeterminate pulmonary nodules includes variants for adults aged 35 or older with a nodule incidentally found on chest CT. The criteria address the appropriateness of the next imaging st

In [3]:
from langchain_core.prompts import ChatPromptTemplate


messages = [
    ("system", f"""你是一名从业 20 年的医疗辅助决策专家，你可以通过用户提供的报告、问题和相关证据进行回答。
     # 你的回答严格遵循以下要求: 
     - 病历事实来自报告，医学判断必须由提供的证据支持
     - 每条医学判断标注对应证据 ID, 来源名称、定位和链接必须取自输入资料
     - 区分“报告缺少条件”和“检索材料不足”；明确还需要确认什么或补查什么
     - 证据正文是项目转述，不能冒充逐字原文引用，也不能编造证据或来源
     - 有证据支持的判断，与待核验的候选线索分别表述
     - 具体的指南名和医学建议，也需要检查是否得到当前材料支持
     - 围绕当前问题列缺失条件，说明每项缺失会影响什么
     - 报告事实、项目转述和原文引用保持清楚的区分
     
     # 你的回答应该分成三个部分：
     - 报告中的已知事实
     - 证据支持的判断
     - 当前不足与下一步
     """),
    ("user", "我是医生，请你根据病历报告和证据卡 (项目转述) 回答问题。\n\n患者的病历报告为: {report}\n\n检索到的证据卡(项目转述) 为: {evidence_context}\n\n问题是: {question}")
]

prompts = ChatPromptTemplate.from_messages(messages) # 为什么要用 ChatPromptTemplate 和 from_messages 函数？与构造常规字符串有什么不同？
inputs = {
    "report": rag_case["report"],
    "question": rag_case["question"],
    "evidence_context": evidence_context
}

full_prompts = prompts.invoke(inputs)
rag_response = model.invoke(full_prompts)

print("模型回答：\n", rag_response.content)
print("Token 用量：", rag_response.usage_metadata)
print("结束原因：", rag_response.response_metadata.get("finish_reason"))


模型回答：
 ## 报告中的已知事实

- **检查背景**：基线低剂量 CT，属于**肺癌筛查项目**（Indication: Baseline low-dose CT performed within a lung cancer screening program）。
- **患者情况**：61 岁成人；无已知原发肿瘤、无免疫抑制（History）。
- **影像发现**：右肺上叶**单发实性肺结节**（Finding: Single solid pulmonary nodule in the right upper lobe）。
- 报告**未提供**：结节大小（直径/体积）、形态学特征（边缘、钙化、密度）、既往 CT 对比情况、吸烟史与 pack-year、以及该筛查项目所采用的报告框架（如 Lung-RADS 分类）。

## 证据支持的判断

1. **本例属于"肺癌筛查 CT"情境，而非"偶然发现结节"情境。** 报告明确写明检查是筛查项目内的基线低剂量 CT。证据 F_SCOPE 指出：Fleischner 2017 针对的是成人（≥35 岁）CT 上**偶然发现**的肺结节，而**肺癌筛查 CT 是独立情境，有其自身的报告框架**（证据 ID：F_SCOPE，Fleischner Society / Radiology，Introduction and Implications for Patient Care，https://pubs.rsna.org/doi/full/10.1148/radiol.2017161659）。因此，决定指南选择的关键事实是**"结节是在筛查 CT 中检出，还是偶然检出"**——本例为前者。

2. **Fleischner 2017 与 ACR 偶然结节 Appropriateness Criteria 均不直接适用于本例的解读。** 证据 AC_INCIDENTAL 说明 ACR 该文件的 Variants 2 和 3 针对的是成人（≥35 岁）**胸部 CT 上偶然发现**的不确定肺结节，讨论的是"下一步影像检查"的适宜性（证据 ID：AC_INCIDENTAL，ACR Appropriateness Criteria，Variants 2 and 3，https://acsearch.acr.org/docs

In [ ]:
selected_ids = rag_case["retrieved_ids"][:3]
all_evidence_ids = selected_ids.copy()
for evidence_id in evidence_df["id"].to_list():
    if evidence_id not in all_evidence_ids:
        all_evidence_ids.append(evidence_id)



top3_context = build_context(selected_ids, evidence_df, sources)
inputs["evidence_context"] = top3_context
top3_response = model.invoke(prompts.invoke(inputs))
all_context = build_context(all_evidence_ids, evidence_df, sources)
inputs["evidence_context"] = all_context
all_response = model.invoke(prompts.invoke(inputs))


print("Top-3 数量：", len(selected_ids))
print("全量数量：", len(all_evidence_ids))
print("全量 ID 唯一数量：", len(set(all_evidence_ids)))
print("原三张卡位置保持一致：", all_evidence_ids[:3] == selected_ids)

for label, result in [("Top-3", top3_response), ("全量", all_response)]:
    print(f"\n===== {label} =====")
    print(result.content)
    print("Token 用量：", result.usage_metadata)
    print("结束原因：", result.response_metadata.get("finish_reason"))

Top-3 数量： 3
全量数量： 15
全量 ID 唯一数量： 15
原三张卡位置保持一致： True

===== Top-3 =====
## 报告中的已知事实

- **检查背景**：基线低剂量 CT，属于**肺癌筛查项目**（Indication: Baseline low-dose CT performed within a lung cancer screening program）。
- **患者情况**：61 岁成人；无已知原发肿瘤、无免疫抑制（History）。
- **影像发现**：右肺上叶**单发实性肺结节**（Finding: Single solid pulmonary nodule in the right upper lobe）。
- 报告**未提供**：结节大小（直径/体积）、形态学特征（边缘、钙化、密度）、既往 CT 对比情况、吸烟史与 pack-year、以及该筛查项目所采用的报告框架（如 Lung-RADS 分类）。

## 证据支持的判断

1. **本例属于"肺癌筛查 CT"情境，而非"偶然发现结节"情境。** 报告明确写明检查是筛查项目内的基线低剂量 CT。证据 F_SCOPE 指出：Fleischner 2017 针对的是成人（≥35 岁）CT 上**偶然发现**的肺结节，而**肺癌筛查 CT 是独立情境，有其自身的报告框架**（F_SCOPE，Fleischner Society / Radiology，Introduction and Implications for Patient Care，https://pubs.rsna.org/doi/full/10.1148/radiol.2017161659）。因此，决定指南选择的关键事实是**"结节是在筛查 CT 中检出，还是在非筛查 CT 上偶然检出"**——本例为前者。

2. **Fleischner 2017 不适用于本例的初始解读。** 依据同上：其适用范围被限定为偶然发现的结节，筛查情境被明确排除在外（F_SCOPE）。

3. **ACR 偶然肺结节 Appropriateness Criteria 同样不匹配本例。** 该标准的 Variant 2/3 针对成人（≥35 岁）在胸部 CT 上**偶然发现**的不确定肺结节，讨论的是"下一步影像检查"的

In [ ]:
rag_case